# BCI for Inner Speech Decoding: Setup and Data Exploration

## Imports

In [ ]:
# Import required libraries
import mne
import warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
import sys
from scipy.signal import butter, filtfilt, iirnotch
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from mne.decoding import CSP


## Set paths

In [ ]:

# Add the Inner Speech Dataset repository to the system path
project_root = Path.cwd()
print(f"Current working directory: {project_root}")
if not (project_root / "Python_Processing").exists():
    project_root = project_root / "Inner_Speech_Dataset-main"
sys.path.insert(0, str(project_root / "Python_Processing"))
print(f"Python_Processing directory added to system path: {sys.path[0]}")

# Import custom functions from the Inner Speech Dataset repository
from lib.data_extractions import (
    extract_data_from_subject,
)
from lib.data_processing import (
    select_time_window,
    transform_for_classificator,
)

from lib.download_helper import (
    _clone_inner_speech,
    get_derivatives,
)


# Set random seed for reproducibility
np.random.seed(23)

# Configure MNE to show only warnings (suppress info messages)
mne.set_log_level(verbose="warning")

# Suppress deprecation and future warnings
warnings.filterwarnings(action="ignore", category=DeprecationWarning)
warnings.filterwarnings(action="ignore", category=FutureWarning)

print("Setup complete! All libraries imported successfully.")

## Clone Dataset

In [ ]:

# Using this path to download the dataset
data_dir = Path().resolve().parent

# Clone just the structure (no files yet). This step is needed to create the
data_dir = _clone_inner_speech(target_dir=data_dir, verbose=False)

# Download the files needed for this script. This can take some minutes.
get_derivatives(
    dataset_path=data_dir,
    subjects="all",
    sessions="all",
    file_types=["eeg","events"],
    verbose=True,
    progress=True,
)


## Parameters

In [ ]:
# Original sampling rate
sfreq = 256

# Target sampling rate
target_sfreq = 128

# Filtering parameters
lowcut = 1.0
highcut = 40.0

notch_freq = 50.0  # use 60 if your environment uses 60Hz mains

# Epoch window
tmin = 0.0
tmax = 2.0

In [ ]:
# Set the hyperparameters for data loading and processing

# Directory containing the dataset (adjust this to your local path)
root_dir = Path.cwd().parent / "ds003626" # Path to the dataset

# Specify which type of data to use
# Options: "EEG" (brain activity), "exg" (physiological signals), or "baseline" (resting state)
datatype = "EEG"

# EEG sampling rate (256 samples per second)
fs = 256

# Select the time window of interest (in seconds)
# The cue appears at t=0, action period is between 1.5s and 3.5s
t_start = 1.5  # Start time after cue
t_end = 3.5  # End time after cue

# Subject to analyze (1-10)
N_S = 1  # We'll start with subject 1

print(
    f"Parameters set: Using {datatype} data from subject {N_S}, analyzing time window {t_start}s to {t_end}s"
)

## Explore the data from subject 1

In [ ]:
# Load all trials for the selected subject
# This combines data from all three experimental blocks
X, Y = extract_data_from_subject(root_dir, N_S, datatype)

print("Data successfully loaded!")
print("\nX (EEG data) shape:", X.shape)
print("  - Number of trials:", X.shape[0])
print("  - Number of EEG channels:", X.shape[1])
print("  - Number of time points per trial:", X.shape[2])
print("  - Total duration per trial:", X.shape[2] / fs, "seconds")

print("\nY (labels) shape:", Y.shape)
print("  - Number of trials:", Y.shape[0])
print("  - Label information columns:", Y.shape[1])

# Check how many trials we have of each condition and direction
conditions = ["Pronounced", "Inner Speech", "Visualized"]
directions = ["Up", "Down", "Right", "Left"]

print("\nNumber of trials per condition:")
for i, condition in enumerate(conditions):
    condition_count = np.sum(Y[:, 2] == i)
    print(f"{condition}: {condition_count}")

print("\nNumber of trials per direction:")
for i, direction in enumerate(directions):
    direction_count = np.sum(Y[:, 1] == i)
    print(f"{direction}: {direction_count}")